In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import os
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight

In [19]:
train_df = pd.read_csv("train_metadata.csv")
test_df = pd.read_csv("test_metadata.csv")

train_files = train_df["ID"].to_list()
labels_train = train_df["label"].to_list()
label_dict = dict(zip(train_df["ID"], train_df["label"]))
test_files = test_df["ID"].to_list()

In [20]:
train_idx = np.random.randint(0, len(train_files))
X0 = io.imread(f"./train/{train_files[train_idx]}")

print(X0.shape)
print(len(train_files))

(370, 368, 3)
28901


In [21]:
L=train_df['label'].unique()
dict_labels = {L[i]:i for i in range(len(L))}
y_train=train_df["label"].map(dict_labels).astype(int)

In [22]:
class ImageDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.label_dict = dict_labels 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['ID']
        img_path = os.path.join(self.root_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        label = y_train.iloc[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

transform = transforms.Compose([
    transforms.Resize((368, 368)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
full_dataset = ImageDataset(df=train_df, root_dir="./train/", transform=transform)

In [23]:
train_indices, val_indices = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)
train_subset = Subset(full_dataset, train_indices)
val_subset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

In [24]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes):
        super(CNNClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.AdaptiveAvgPool2d((7, 7))
        )
        self.classifier= nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*7*7, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features=self.features(x)
        class_x = self.classifier(features)
        return class_x
    

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(L)
model = CNNClassifier(num_classes).to(device)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10

In [26]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Training, epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Validation simple
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validation, epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, Val Acc: {100 * correct / total:.2f}%")

Validation, epoch 1: 100%|██████████| 181/181 [01:44<00:00,  1.73it/s]


Epoch [1/10], Loss: 2.4205, Val Acc: 53.21%


Validation, epoch 2: 100%|██████████| 181/181 [01:22<00:00,  2.20it/s]


Epoch [2/10], Loss: 2.0263, Val Acc: 57.34%


Validation, epoch 3: 100%|██████████| 181/181 [01:46<00:00,  1.70it/s]


...

Epoch [8/10], Loss: 1.4877, Val Acc: 69.54%


Validation, epoch 9: 100%|██████████| 181/181 [01:26<00:00,  2.10it/s]


Epoch [9/10], Loss: 1.4443, Val Acc: 68.03%


Validation, epoch 10: 100%|██████████| 181/181 [01:28<00:00,  2.04it/s]

Epoch [10/10], Loss: 1.3936, Val Acc: 64.61%


In [27]:
model.eval() # 1. Mode évaluation (désactive le Dropout)
predictions = []

# Utilise le même objet transform que pour le train
with torch.no_grad(): # 2. Désactive le calcul des gradients
    for i in tqdm(range(len(test_files))):
        # 3. Charger et transformer proprement
        img_path = f'./test/{test_files[i]}'
        image = Image.open(img_path).convert('RGB')
        image = transform(image) # Applique Resize, ToTensor, Normalize
        
        # 4. Ajouter la dimension de batch (1, C, H, W) et envoyer sur GPU
        image = image.unsqueeze(0).to(device)
        
        # 5. Passer dans le modèle
        outputs = model(image)
        
        # 6. Convertir les logits en index de classe
        _, predicted = torch.max(outputs, 1)
        predictions.append(predicted.item())

inv_dict_labels = {v: k for k, v in dict_labels.items()}
test_df['label'] = [inv_dict_labels[p] for p in predictions]

# 8. Sauvegarde
test_df.to_csv('submission_CNN_only.csv', index=False)

100%|██████████| 9634/9634 [02:53<00:00, 55.53it/s]
